# Review copy

Unified Phase 0/Phase 1 runner and damped-refiner diagnostic, original cells 29–31. Extracted from the Drive notebook experiments_and _tests.ipynb without changing code or Markdown. Outputs and execution counts were removed to reduce file size. This part is for review and is not standalone. Original cell order is preserved.

In [ ]:
# @title
# ==============================================================================
# UNIFIED COLAB RUNNER: AUTO-RESOLVES /content/dataset OR GOOGLE DRIVE
# Runs Phase 0 Sanity + Phase 1 Prior-Conditioned Refiner on GPU or CPU
# ==============================================================================
import csv
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
print(f"Active Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})")

# ==============================================================================
# 1. BULLETPROOF DATASET & CAMERA PARAMETER RESOLUTION
# ==============================================================================
CAPTURE_ID = "m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands"

def locate_dataset_files():
    search_roots = [Path("/content/dataset"), Path("/content"), Path.cwd()]
    if Path("/content/drive/MyDrive").exists():
        search_roots.append(Path("/content/drive/MyDrive"))

    manifest_path = None
    cam_params_path = None
    file_map = {}

    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            name_low = p.name.lower()
            if name_low == "manifest.csv" and manifest_path is None:
                manifest_path = p
            elif name_low in ("cam_params.json", "cameras.json", "calibration.json") and cam_params_path is None:
                cam_params_path = p
            elif p.suffix.lower() in (".png", ".jpg", ".jpeg", ".json"):
                if p.name not in file_map:
                    file_map[p.name] = p

    if manifest_path is None:
        # Try mounting Google Drive only if manifest wasn't found locally in /content
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            for p in Path("/content/drive/MyDrive").rglob("*"):
                if not p.is_file():
                    continue
                if p.name.lower() == "manifest.csv" and manifest_path is None:
                    manifest_path = p
                elif p.name.lower() == "cam_params.json" and cam_params_path is None:
                    cam_params_path = p
                elif p.suffix.lower() in (".png", ".jpg", ".jpeg", ".json"):
                    if p.name not in file_map:
                        file_map[p.name] = p
        except Exception:
            pass

    if manifest_path is None:
        raise FileNotFoundError("Could not find manifest.csv anywhere under /content or /content/drive!")

    return manifest_path, cam_params_path, file_map

MANIFEST_PATH, CAMERA_PATH, FILE_MAP = locate_dataset_files()
BASE_DIR = MANIFEST_PATH.parent
print(f"Found Manifest : {MANIFEST_PATH}")
print(f"Indexed Files  : {len(FILE_MAP):,} images/JSONs")

def resolve_asset(rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    for cand in [
        Path(rel_clean),
        BASE_DIR / rel_clean,
        BASE_DIR.parent / rel_clean,
        BASE_DIR.parent.parent / rel_clean,
    ]:
        if cand.is_file():
            return cand
    if rel_clean.startswith("dataset/"):
        c = BASE_DIR / rel_clean[len("dataset/"):]
        if c.is_file():
            return c
    fname = Path(rel_clean).name
    if fallback_subdir:
        for cand in [BASE_DIR / fallback_subdir / fname, BASE_DIR.parent / fallback_subdir / fname]:
            if cand.is_file():
                return cand
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve asset: {rel_path}")

# ==============================================================================
# 2. PHASE 0 CONSTANTS, CAMERA CALIBRATION & IN-MEMORY DATASET CACHE
# ==============================================================================
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
FINGER_CHAINS = [
    [0, 1, 2, 3, 4], [0, 5, 6, 7, 8], [0, 9, 10, 11, 12],
    [0, 13, 14, 15, 16], [0, 17, 18, 19, 20],
]

IMAGE_W, IMAGE_H = 128, 96
SEED = 20251018
TRAIN_SEGMENTS = 44
VAL_SEGMENTS = 12
OVERFIT_SAMPLES = 64
OVERFIT_MAX_EPOCHS = 250
TRAIN_EPOCHS = 20
PHASE1_EPOCHS = 25
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4

def get_camera_400485_params(sample_world_pts: np.ndarray, img_w: int, img_h: int):
    """Loads cam_params.json if present; otherwise calibrates camera-frame pinhole projection."""
    if CAMERA_PATH is not None and CAMERA_PATH.is_file():
        with CAMERA_PATH.open("r", encoding="utf-8") as f:
            cam_dict = json.load(f)
        cam = cam_dict.get("400485", next(iter(cam_dict.values())))
        R = np.asarray(cam["R"], dtype=np.float32)
        t = np.asarray(cam["t"], dtype=np.float32)
        focal = np.asarray(cam["focal"], dtype=np.float32)
        princpt = np.asarray(cam["princpt"], dtype=np.float32)
        print(f"Loaded official camera 400485 calibration from {CAMERA_PATH}")
        return R, t, focal, princpt

    # Automatic pinhole calibration when running on standalone /content/dataset without cam_params.json:
    # Centers the 3D hand cluster at Z=550mm along the optical axis of the w x h image
    print("[INFO] cam_params.json not in /content/dataset; using auto-centered pinhole calibration for 400485.")
    R = np.eye(3, dtype=np.float32)
    mean_xyz = sample_world_pts.mean(axis=(0, 1)).astype(np.float32)
    t = np.array([-mean_xyz[0], -mean_xyz[1], 550.0 - mean_xyz[2]], dtype=np.float32)
    fx = float(img_w) * 1.8
    fy = float(img_h) * 1.8
    focal = np.array([fx, fy], dtype=np.float32)
    princpt = np.array([img_w * 0.5, img_h * 0.5], dtype=np.float32)
    return R, t, focal, princpt

def crop_affine(uv, width, height, margin=1.4, out_w=None, out_h=None):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    out_w = IMAGE_W if out_w is None else out_w
    out_h = IMAGE_H if out_h is None else out_h
    return np.asarray([
        [out_w / side, 0.0, -left * out_w / side],
        [0.0, out_h / side, -top * out_h / side],
    ], np.float32)

def make_channels(gray_u8: np.ndarray) -> np.ndarray:
    gray = gray_u8.astype(np.float32) / 255.0
    gray_255 = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray_255, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray_255, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.sqrt(gx * gx + gy * gy)
    return np.stack([gray, np.clip(mag, 0.0, 1.0)], axis=0).astype(np.float32)

# Self-healing RAM cache so re-running never re-reads 5,114 images from disk
if "CACHED_RECORDS" not in globals() or len(globals().get("CACHED_RECORDS", [])) == 0:
    t_load = time.perf_counter()
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r.get("camera_id", "400485") == "400485"]

    print(f"Ingesting {len(rows):,} frames and 3D keypoints into memory...")
    raw_world_list = []
    valid_rows = []
    first_img_shape = None

    for r in rows:
        try:
            kp_p = resolve_asset(r["orig_kp_right"], "keypoints_right")
            img_p = resolve_asset(r["image_path"], "images")
            with kp_p.open("r", encoding="utf-8") as kf:
                kp_w = np.asarray(json.load(kf), dtype=np.float32)
            if first_img_shape is None:
                im0 = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
                if im0 is None:
                    continue
                first_img_shape = (im0.shape[1], im0.shape[0])
            raw_world_list.append(kp_w)
            valid_rows.append((r, img_p))
        except Exception:
            continue

    raw_world_arr = np.stack(raw_world_list, axis=0)
    IMG_ORIG_W, IMG_ORIG_H = first_img_shape
    CAM_R, CAM_T, CAM_FOCAL, CAM_PRINCPT = get_camera_400485_params(raw_world_arr, IMG_ORIG_W, IMG_ORIG_H)

    CACHED_RECORDS = []
    CACHED_GRAY_BUFFERS = []
    CACHED_BUFFER_AFFINES = []
    CACHED_ROOTS = np.empty((len(valid_rows), 3), dtype=np.float32)

    for idx, (row, img_p) in enumerate(valid_rows):
        gray = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            continue
        h, w = gray.shape[:2]
        kp_world = raw_world_arr[idx]
        kp_cam_mm = kp_world @ CAM_R.T + CAM_T

        selected_cam = kp_cam_mm[KEEP_21_INDICES]
        z = np.maximum(selected_cam[:, 2], 1.0)
        selected_uv = np.stack([
            CAM_FOCAL[0] * selected_cam[:, 0] / z + CAM_PRINCPT[0],
            CAM_FOCAL[1] * selected_cam[:, 1] / z + CAM_PRINCPT[1],
        ], axis=1).astype(np.float32)

        # If using fallback intrinsics (no cam_params.json), crop from foreground hand silhouette
        if CAMERA_PATH is None:
            ys, xs = np.where(gray > 18)
            if len(xs) > 60:
                umin, umax = float(np.percentile(xs, 2)), float(np.percentile(xs, 98))
                vmin, vmax = float(np.percentile(ys, 2)), float(np.percentile(ys, 98))
                # Map 3D keypoints into the detected foreground bounding box
                s3_min, s3_max = selected_cam[:, :2].min(axis=0), selected_cam[:, :2].max(axis=0)
                s3_span = np.maximum(s3_max - s3_min, 10.0)
                selected_uv[:, 0] = umin + (selected_cam[:, 0] - s3_min[0]) / s3_span[0] * (umax - umin)
                selected_uv[:, 1] = vmin + (selected_cam[:, 1] - s3_min[1]) / s3_span[1] * (vmax - vmin)

        root_relative = selected_cam - selected_cam[0:1]
        affine = crop_affine(selected_uv, w, h, margin=1.4, out_w=IMAGE_W, out_h=IMAGE_H)
        crop = cv2.warpAffine(
            gray, affine, (IMAGE_W, IMAGE_H),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        uv_crop = selected_uv @ affine[:, :2].T + affine[:, 2]

        buf_aff = crop_affine(selected_uv, w, h, margin=3.5, out_w=384, out_h=288)
        buf_gray = cv2.warpAffine(
            gray, buf_aff, (384, 288),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )

        CACHED_RECORDS.append({
            "frame_id": row.get("frame_id", str(idx)),
            "segment": row["segment"],
            "uv_image": selected_uv,
            "crop_gray": crop,
            "uv_crop": uv_crop.astype(np.float32),
            "y3d": root_relative.astype(np.float32),
        })
        CACHED_GRAY_BUFFERS.append(buf_gray)
        CACHED_BUFFER_AFFINES.append(buf_aff)
        CACHED_ROOTS[len(CACHED_RECORDS) - 1] = selected_cam[0]

    CACHED_BUFFER_AFFINES = np.stack(CACHED_BUFFER_AFFINES)
    CACHED_ROOTS = CACHED_ROOTS[:len(CACHED_RECORDS)]
    print(f"Staged {len(CACHED_RECORDS):,} frames into memory in {time.perf_counter() - t_load:.1f}s.")
else:
    print(f"Reusing {len(CACHED_RECORDS):,} frames already cached in memory.")

# ==============================================================================
# 3. MODELS & HELPERS (PHASE 0 LCF + PHASE 1 POSE REFINER)
# ==============================================================================
def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    batch = bone_vectors.shape[0]
    joints = torch.zeros((batch, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints

class WebcamAlignedLCFTracker(nn.Module):
    def __init__(self, canonical_lens: torch.Tensor):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.proj2d_head = nn.Linear(512, 21 * 2)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x: torch.Tensor):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7)
        latent_frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).reshape(batch, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(latent_frame, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for finger_idx in range(4):
            bone_offset = finger_offsets[finger_idx]
            phalanx_offset = 3 + finger_idx * 3
            bone_dirs[:, bone_offset] = meta_cam_u[:, finger_idx + 1]
            bone_dirs[:, bone_offset + 1:bone_offset + 4] = phalanx_cam_u[:, phalanx_offset:phalanx_offset + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        kp2d = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, bone_dirs, kp2d

class PoseRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.1)
        )
        self.delta_head = nn.Linear(512, 21 * 3)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        return self.delta_head(feat).reshape(-1, 21, 3)

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        l_pos = torch.norm(pred - target, dim=-1).mean()
        pred_bones = pred[:, [c for _, c in BONE_PAIRS]] - pred[:, [p for p, _ in BONE_PAIRS]]
        target_bones = target[:, [c for _, c in BONE_PAIRS]] - target[:, [p for p, _ in BONE_PAIRS]]
        pred_len = torch.norm(pred_bones, dim=-1)
        target_len = torch.norm(target_bones, dim=-1)
        l_len = F.l1_loss(pred_len, target_len)
        eps = 1e-7
        pred_u = pred_bones / (pred_len.unsqueeze(-1) + eps)
        target_u = target_bones / (target_len.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((pred_u * target_u).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

def rotate_camera_xy(y3d: np.ndarray, angle_rad: float) -> np.ndarray:
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    out = y3d.copy()
    x, y = y3d[:, 0].copy(), y3d[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out

def augment_one(gray_u8: np.ndarray, uv: np.ndarray, y3d: np.ndarray, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    center = (IMAGE_W * 0.5, IMAGE_H * 0.5)
    matrix = cv2.getRotationMatrix2D(center, angle_deg, scale).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray_u8, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(warped.astype(np.float32) * brightness + offset, 0, 255).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    y2d_aug = np.empty_like(uv_aug, dtype=np.float32)
    y2d_aug[:, 0] = 2.0 * uv_aug[:, 0] / IMAGE_W - 1.0
    y2d_aug[:, 1] = 2.0 * uv_aug[:, 1] / IMAGE_H - 1.0
    return make_channels(warped), y3d_aug, y2d_aug

def normalize_crop_points(uv: np.ndarray) -> np.ndarray:
    out = np.empty_like(uv, dtype=np.float32)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out

def score(pred: np.ndarray, target: np.ndarray) -> tuple[float, list[float]]:
    distance = np.linalg.norm(pred - target, axis=-1)
    return float(distance.mean()), [float(x) for x in distance[:, FINGERTIP_JOINTS].mean(axis=0)]

def predict_metrics(model: nn.Module, x: torch.Tensor, target: np.ndarray):
    preds = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(x), 128):
            pred, _, _ = model(x[start:start + 128].to(DEVICE))
            preds.append(pred.cpu().numpy())
    return score(np.concatenate(preds, axis=0), target)

def train_tracker(model, train_records, train_indices, train_y3d, train_y2d, x_base, epochs, augment, val_x=None, val_y3d=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=1e-4)
    criterion = MetricLossEvaluator()
    rng = np.random.default_rng(SEED + (1 if augment else 2))
    val_history = []
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(len(train_indices))
        for offset in range(0, len(order), BATCH_SIZE):
            local_batch = order[offset:offset + BATCH_SIZE]
            indices = train_indices[local_batch]
            if augment:
                batch = [
                    augment_one(train_records[int(i)]["crop_gray"], train_records[int(i)]["uv_crop"], train_y3d[local_pos], rng)
                    for local_pos, i in zip(local_batch, indices)
                ]
                bx = torch.from_numpy(np.stack([item[0] for item in batch])).to(DEVICE)
                by3d = torch.from_numpy(np.stack([item[1] for item in batch])).to(DEVICE)
                by2d = torch.from_numpy(np.stack([item[2] for item in batch])).to(DEVICE)
            else:
                bx = x_base[indices].to(DEVICE)
                by3d = torch.from_numpy(train_y3d[local_batch]).to(DEVICE)
                by2d = torch.from_numpy(train_y2d[local_batch]).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred3d, _, pred2d = model(bx)
            loss = criterion(pred3d, by3d) + 15.0 * F.smooth_l1_loss(pred2d, by2d)
            loss.backward()
            optimizer.step()

        scheduler.step()
        if val_x is not None and val_y3d is not None and epoch % 2 == 0:
            mpjpe, tips = predict_metrics(model, val_x, val_y3d)
            val_history.append((epoch, mpjpe, tips))
        if not augment:
            train_mpjpe, _ = predict_metrics(model, x_base[train_indices], train_y3d)
            if train_mpjpe < 3.0:
                break
    return val_history, time.perf_counter() - started

# ==============================================================================
# 4. EXECUTE PHASE 0 SANITY + PHASE 1 PRIOR-CONDITIONED REFINEMENT
# ==============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

segments = sorted({r["segment"] for r in CACHED_RECORDS})
split_rng = np.random.default_rng(SEED)
shuffled = np.asarray(segments, dtype=object)
split_rng.shuffle(shuffled)
train_segments = set(shuffled[:TRAIN_SEGMENTS].tolist())
val_segments = set(shuffled[TRAIN_SEGMENTS:].tolist())

train_idx = np.asarray([i for i, r in enumerate(CACHED_RECORDS) if r["segment"] in train_segments], dtype=np.int64)
val_idx = np.asarray([i for i, r in enumerate(CACHED_RECORDS) if r["segment"] in val_segments], dtype=np.int64)

x_all = torch.from_numpy(np.stack([make_channels(r["crop_gray"]) for r in CACHED_RECORDS]))
y_all = np.stack([r["y3d"] for r in CACHED_RECORDS]).astype(np.float32)
y2d_all = np.stack([normalize_crop_points(r["uv_crop"]) for r in CACHED_RECORDS]).astype(np.float32)

train_y3d, val_y3d = y_all[train_idx], y_all[val_idx]
train_y2d = y2d_all[train_idx]
val_x = x_all[val_idx]

print("\n" + "=" * 95)
print(" PHASE 0: BASELINES, OVERFIT-64 DIAGNOSTIC & 2-CH WEBCAM-ALIGNED LCF")
print("=" * 95)

mean_pose = train_y3d.mean(axis=0, keepdims=True)
mean_mpjpe, mean_tips = score(np.repeat(mean_pose, len(val_idx), axis=0), val_y3d)

with torch.inference_mode():
    low_res = F.adaptive_avg_pool2d(x_all.to(DEVICE), (12, 16)).flatten(1)
    low_train, low_val = low_res[train_idx], low_res[val_idx]
    norm_train = (low_train * low_train).sum(dim=1)
    nn_indices = []
    for start in range(0, len(low_val), 64):
        chunk = low_val[start:start + 64]
        dists = (chunk * chunk).sum(dim=1, keepdim=True) + norm_train.unsqueeze(0) - 2.0 * (chunk @ low_train.T)
        nn_indices.extend(torch.argmin(dists, dim=1).tolist())
nn_mpjpe, nn_tips = score(train_y3d[np.asarray(nn_indices, dtype=np.int64)], val_y3d)

canonical_lens = torch.from_numpy(np.stack([
    np.linalg.norm(train_y3d[:, c] - train_y3d[:, p], axis=1).mean() for p, c in BONE_PAIRS
]).astype(np.float32))

overfit_rng = np.random.default_rng(SEED + 64)
overfit_local = overfit_rng.choice(len(train_idx), size=min(OVERFIT_SAMPLES, len(train_idx)), replace=False)
overfit_indices = train_idx[overfit_local]
overfit_model = WebcamAlignedLCFTracker(canonical_lens).to(DEVICE)
local_records = [CACHED_RECORDS[i] for i in overfit_indices]
_, overfit_sec = train_tracker(
    overfit_model, local_records, np.arange(len(overfit_indices), dtype=np.int64),
    y_all[overfit_indices], y2d_all[overfit_indices], x_all[overfit_indices],
    epochs=OVERFIT_MAX_EPOCHS, augment=False,
)
overfit_mpjpe, overfit_tips = predict_metrics(overfit_model, x_all[overfit_indices], y_all[overfit_indices])
overfit_status = "PASS" if overfit_mpjpe < 3.0 else "FAIL"

direct = WebcamAlignedLCFTracker(canonical_lens).to(DEVICE)
val_history, train_sec = train_tracker(
    direct, CACHED_RECORDS, train_idx, train_y3d, train_y2d, x_all,
    epochs=TRAIN_EPOCHS, augment=True, val_x=val_x, val_y3d=val_y3d,
)

def metric_line(method, split, mpjpe, tips):
    tip_text = " ".join(f"{name}={val:6.2f}" for name, val in zip(FINGER_NAMES, tips))
    return f"{method:<22} {split:<12} {mpjpe:8.2f} mm | {tip_text}"

print(metric_line("Mean pose", "val", mean_mpjpe, mean_tips))
print(metric_line("Nearest neighbor", "val", nn_mpjpe, nn_tips))
print(metric_line(f"Overfit64 ({overfit_status})", "train", overfit_mpjpe, overfit_tips))
for ep, mpjpe, tips in val_history:
    print(metric_line("WebcamAlignedLCF", f"epoch {ep:02d}", mpjpe, tips))

# ==============================================================================
# 5. PHASE 1: PRIOR-CONDITIONED POSE REFINER
# ==============================================================================
print("\n" + "=" * 95)
print(" PHASE 1: PRIOR-CONDITIONED ITERATIVE POSE REFINEMENT")
print("=" * 95)

def project_pts(points_cam, focal, principal):
    z = np.maximum(points_cam[:, 2], 1.0)
    return np.stack((focal[0] * points_cam[:, 0] / z + principal[0],
                     focal[1] * points_cam[:, 1] / z + principal[1]), axis=1).astype(np.float32)

def rotation_matrix(axis, angle):
    axis = np.asarray(axis, np.float64)
    axis /= max(np.linalg.norm(axis), 1e-12)
    x, y, z = axis
    c, s, d = np.cos(angle), np.sin(angle), 1.0 - np.cos(angle)
    return np.asarray([
        [c + x*x*d, x*y*d - z*s, x*z*d + y*s],
        [y*x*d + z*s, c + y*y*d, y*z*d - x*s],
        [z*x*d - y*s, z*y*d + x*s, c + z*z*d],
    ], np.float32)

def synthesize_prior(gt, rng, severity=None):
    s = float(rng.uniform(0.2, 1.0) if severity is None else severity)
    prior = gt.copy()
    for chain in FINGER_CHAINS:
        for hinge_pos in range(len(chain) - 1):
            hinge = chain[hinge_pos]
            subtree = chain[hinge_pos + 1:]
            mat = rotation_matrix(rng.normal(size=3), np.deg2rad(rng.uniform(-25.0, 25.0) * s))
            prior[subtree] = (prior[subtree] - prior[hinge]) @ mat.T + prior[hinge]
    global_rot = rotation_matrix(rng.normal(size=3), np.deg2rad(rng.uniform(-10, 10)))
    prior = prior @ global_rot.T
    prior += rng.normal(0.0, 8.0, size=(1, 3)).astype(np.float32)
    return prior.astype(np.float32)

def channels_from_prior(gray, uv, affine):
    crop = cv2.warpAffine(gray, affine, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    xy = uv @ affine[:, :2].T + affine[:, 2]
    lines = np.zeros((IMAGE_H, IMAGE_W), np.uint8)
    dots = np.zeros_like(lines)
    points = np.rint(xy).astype(int)
    for parent, child in BONE_PAIRS:
        cv2.line(lines, tuple(points[parent]), tuple(points[child]), 255, 2, cv2.LINE_AA)
    for point in points:
        cv2.circle(dots, tuple(point), 3, 255, -1, cv2.LINE_AA)
    base = make_channels(crop)
    return np.concatenate((base, lines[None].astype(np.float32) / 255.0, dots[None].astype(np.float32) / 255.0), axis=0), xy

def pose_input(index, prior):
    uv_img = project_pts(CACHED_ROOTS[index:index+1] + prior, CAM_FOCAL, CAM_PRINCPT)
    affine = crop_affine(uv_img, IMG_ORIG_W, IMG_ORIG_H)
    src = np.eye(3, dtype=np.float32); src[:2] = CACHED_BUFFER_AFFINES[index]
    dst = np.eye(3, dtype=np.float32); dst[:2] = affine
    mapping = dst @ np.linalg.inv(src)
    prior_gray = cv2.warpAffine(CACHED_GRAY_BUFFERS[index], mapping[:2], (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    inp, uv_crop = channels_from_prior(prior_gray, uv_img, affine)
    return inp, uv_crop, affine

def make_sample(index, prior, aug_rng=None):
    inp, uv_crop, affine = pose_input(index, prior)
    gt = CACHED_RECORDS[index]["y3d"]
    if aug_rng is None:
        return inp, prior.astype(np.float32), gt, normalize_crop_points(uv_crop), CACHED_ROOTS[index], affine, np.asarray([[1, 0, 0], [0, 1, 0]], np.float32)
    gray_crop = np.clip(inp[0] * 255.0, 0, 255).astype(np.uint8)
    gt_uv_crop = CACHED_RECORDS[index]["uv_image"] @ affine[:, :2].T + affine[:, 2]
    base, gt_aug, uv_norm = augment_one(gray_crop, gt_uv_crop, gt, aug_rng)
    uv_aug = uv_norm.copy()
    uv_aug[:, 0] = (uv_aug[:, 0] + 1) * IMAGE_W * 0.5
    uv_aug[:, 1] = (uv_aug[:, 1] + 1) * IMAGE_H * 0.5
    design = np.column_stack((gt_uv_crop, np.ones(len(gt_uv_crop))))
    affine_t, *_ = np.linalg.lstsq(design, uv_aug, rcond=None)
    mat = affine_t.T.astype(np.float32)
    angle = float(np.arctan2(mat[0, 1], mat[0, 0]))
    prior_aug = rotate_camera_xy(prior, angle).astype(np.float32)
    lines = cv2.warpAffine(inp[2], mat, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    dots = cv2.warpAffine(inp[3], mat, (IMAGE_W, IMAGE_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return np.concatenate((base, lines[None], dots[None]), axis=0), prior_aug, gt_aug, uv_norm, CACHED_ROOTS[index], affine, mat

refiner = PoseRefiner().to(DEVICE)
optimizer = torch.optim.AdamW(refiner.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-4)
rng = np.random.default_rng(SEED + 100)
focal_t = torch.from_numpy(CAM_FOCAL).to(DEVICE)
princ_t = torch.from_numpy(CAM_PRINCPT).to(DEVICE)

for epoch in range(1, PHASE1_EPOCHS + 1):
    refiner.train()
    order = np.random.permutation(train_idx)
    for start in range(0, len(order), BATCH_SIZE):
        ids = order[start:start+BATCH_SIZE]
        samples = [make_sample(int(idx), synthesize_prior(y_all[idx], rng), aug_rng=rng) for idx in ids]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        pri = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        tgt = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)
        uv_tgt = torch.from_numpy(np.stack([s[3] for s in samples])).to(DEVICE)
        roots_b = torch.from_numpy(np.stack([s[4] for s in samples])).to(DEVICE)
        crop_aff = torch.from_numpy(np.stack([s[5] for s in samples])).to(DEVICE)
        aug_aff = torch.from_numpy(np.stack([s[6] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        corrected = pri + refiner(bx)
        loss3d = F.l1_loss(corrected, tgt)

        angles = torch.atan2(aug_aff[:, 0, 1], aug_aff[:, 0, 0])
        c, s = torch.cos(angles)[:, None], torch.sin(angles)[:, None]
        unrot = corrected.clone()
        x, y = corrected[..., 0], corrected[..., 1]
        unrot[..., 0] = c * x - s * y
        unrot[..., 1] = s * x + c * y
        camera_points = unrot + roots_b[:, None, :]
        z = camera_points[..., 2].clamp_min(1.0)
        uv = torch.stack((focal_t[0] * camera_points[..., 0] / z + princ_t[0],
                          focal_t[1] * camera_points[..., 1] / z + princ_t[1]), dim=-1)
        uv_h = torch.cat((uv, torch.ones_like(uv[..., :1])), dim=-1)
        crop_h = torch.bmm(uv_h, crop_aff.transpose(1, 2))
        aug_h = torch.cat((crop_h, torch.ones_like(crop_h[..., :1])), dim=-1)
        pred_crop = torch.bmm(aug_h, aug_aff.transpose(1, 2))
        pred_norm = pred_crop.clone()
        pred_norm[..., 0] = 2.0 * pred_crop[..., 0] / IMAGE_W - 1.0
        pred_norm[..., 1] = 2.0 * pred_crop[..., 1] / IMAGE_H - 1.0
        loss2d = F.smooth_l1_loss(pred_norm, uv_tgt)
        loss = loss3d + 15.0 * loss2d
        loss.backward()
        optimizer.step()
    scheduler.step()

refiner.eval()
direct.eval()

def run_refiner_step(inp, prior):
    with torch.inference_mode():
        delta = refiner(torch.from_numpy(inp[None]).to(DEVICE))[0].cpu().numpy()
    return (prior + delta).astype(np.float32)

for severity in (0.3, 0.6, 1.0):
    rng_eval = np.random.default_rng(SEED + int(severity * 1000))
    before, after = [], []
    for idx in val_idx:
        idx = int(idx)
        prior = synthesize_prior(y_all[idx], rng_eval, severity)
        inp, _, _ = pose_input(idx, prior)
        before.append(prior)
        after.append(run_refiner_step(inp, prior))
    bscore, _ = score(np.stack(before), val_y3d)
    ascore, tips = score(np.stack(after), val_y3d)
    tip_str = " ".join(f"{n}={v:.2f}" for n, v in zip(FINGER_NAMES, tips))
    print(f"Synthetic s={severity:.1f}          held-out | After: {ascore:6.2f} mm (Before: {bscore:6.2f} mm) | {tip_str}")

with torch.inference_mode():
    priors = np.stack([direct(x_all[int(idx):int(idx)+1].to(DEVICE))[0][0].cpu().numpy() for idx in val_idx])

for iteration in range(4):
    current = priors.copy() if iteration == 0 else refined_stack
    scores, tips = score(current, val_y3d)
    tip_str = " ".join(f"{n}={v:.2f}" for n, v in zip(FINGER_NAMES, tips))
    print(f"Realistic iter {iteration}         held-out | After: {scores:6.2f} mm                     | {tip_str}")
    if iteration < 3:
        refined_stack = np.stack([run_refiner_step(pose_input(int(idx), pr)[0], pr) for idx, pr in zip(val_idx, current)])

# Save combined checkpoint for local live webcam studio
torch.save({
    "direct_state_dict": direct.cpu().state_dict(),
    "refiner_state_dict": refiner.cpu().state_dict(),
    "bone_lens": canonical_lens.cpu(),
}, "/content/phase1_combined_tracker.pt")
print("\nSaved combined Phase 0 + Phase 1 checkpoint to /content/phase1_combined_tracker.pt")

In [ ]:
# @title
# =============================================================================
# SELF-CONTAINED COLAB RUNNER
# - Works after a Colab runtime/session reset
# - Resolves flattened keypoint folders such as keypoints_right/
# - Loads and evaluates an older Phase 0/1 checkpoint when available
# - Trains segment-wise out-of-fold direct models
# - Trains and evaluates the new uncertainty-gated refiner
# =============================================================================

import csv
import json
import os
import random
import statistics
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------ Configuration -------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
CHECKPOINT_NAMES = (
    "phase1_combined_tracker.pt",
    "phase0_tracker.pt",
    "phase1_refiner.pt",
)

SEED = 20251018
TRAIN_SEGMENTS, VAL_SEGMENTS = 44, 12
N_FOLDS = 4
TRAIN_EPOCHS = 20
REFINER_EPOCHS = 25
BATCH_SIZE = 64
LR, WEIGHT_DECAY = 1.2e-3, 1e-4
IMAGE_W, IMAGE_H = 128, 96

print(
    f"Active compute device: {DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})"
)

# ----------------------------- Locate the data -------------------------------

def manifests_under(root):
    if not root.exists():
        return []
    return list(root.rglob("manifest.csv"))

def locate_manifest():
    for root in (DATASET_DIR, Path("/content"), Path.cwd()):
        hits = manifests_under(root)
        if hits:
            return next(
                (p for p in hits if "mugsy_subset" in str(p).lower()),
                hits[0],
            )

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped or unavailable: {exc}")

    hits = manifests_under(DRIVE_ROOT)
    if hits:
        return next(
            (p for p in hits if "mugsy_subset" in str(p).lower()),
            hits[0],
        )

    raise FileNotFoundError(
        "Could not find manifest.csv under /content or Google Drive. "
        "Set DATASET_DIR to the extracted dataset folder and rerun."
    )

MANIFEST_PATH = locate_manifest()
SUBSET_DIR = MANIFEST_PATH.parent

def find_capture_root(subset_dir):
    for parent in (subset_dir, *subset_dir.parents):
        if (
            (parent / "Mugsy_cameras" / "cam_params.json").is_file()
            or (parent / "orig_fits").is_dir()
            or (parent / "keypoints_right").is_dir()
        ):
            return parent
    return subset_dir

CAPTURE_DIR = find_capture_root(SUBSET_DIR)
CAMERA_PATH = CAPTURE_DIR / "Mugsy_cameras" / "cam_params.json"

if not CAMERA_PATH.is_file():
    for root in (SUBSET_DIR.parent, SUBSET_DIR.parent.parent, Path("/content")):
        if root.exists():
            hits = list(root.rglob("cam_params.json"))
            if hits:
                CAMERA_PATH = next(
                    (p for p in hits if p.parent.name.lower() == "mugsy_cameras"),
                    hits[0],
                )
                break

if not CAMERA_PATH.is_file():
    # The extracted subset may store the calibration file directly beside the
    # manifest rather than in Mugsy_cameras/.
    nearby = [
        SUBSET_DIR / "cam_params.json",
        SUBSET_DIR.parent / "cam_params.json",
        SUBSET_DIR.parent.parent / "cam_params.json",
    ]
    CAMERA_PATH = next((p for p in nearby if p.is_file()), CAMERA_PATH)

if not CAMERA_PATH.is_file():
    raise FileNotFoundError(
        "Could not locate cam_params.json. Set CAMERA_PATH manually."
    )

print(f"Manifest:     {MANIFEST_PATH}")
print(f"Capture root: {CAPTURE_DIR}")
print(f"Calibration:  {CAMERA_PATH}")

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

rows = [r for r in all_rows if r.get("camera_id", "400485") == "400485"]
if not rows:
    raise RuntimeError("The manifest has no camera_id=400485 rows.")

# Build a filename index once. It may contain duplicate entries because
# overlapping search roots can index the same path; deduplicate them below.
ASSET_INDEX = {}
for root in dict.fromkeys((CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent)):
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {
            ".json", ".png", ".jpg", ".jpeg"
        }:
            ASSET_INDEX.setdefault(path.name.lower(), set()).add(
                path.resolve()
            )

def resolve_asset(rel):
    rel = str(rel).replace("\\", "/").strip()
    rel_path = Path(rel)
    parts = rel_path.parts

    # First try the manifest path literally and relative to likely roots.
    candidates = [
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
        SUBSET_DIR.parent.parent / rel,
    ]
    if parts and parts[0] == CAPTURE_DIR.name:
        candidates.append(CAPTURE_DIR.joinpath(*parts[1:]))

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Try the original relative path against each ancestor of the manifest.
    for ancestor in (SUBSET_DIR, *SUBSET_DIR.parents):
        candidate = ancestor.joinpath(*parts)
        if candidate.is_file():
            return candidate.resolve()

    # Flattened export fallback:
    #   orig_fits/right/Keypoints/foo.json -> keypoints_right/foo.json
    #   orig_fits/left/Keypoints/foo.json  -> keypoints_left/foo.json
    side = None
    lower_parts = [p.lower() for p in parts]
    if "right" in lower_parts:
        side = "right"
    elif "left" in lower_parts:
        side = "left"

    basename = rel_path.name.lower()
    hits = ASSET_INDEX.get(basename, set())

    if side is not None:
        side_folder = f"keypoints_{side}"
        side_hits = {
            p for p in hits
            if side_folder in [part.lower() for part in p.parts]
        }
        if len(side_hits) == 1:
            return next(iter(side_hits))
        if len(side_hits) > 1:
            # Prefer the exact flattened folder directly under the capture root.
            exact = {
                p for p in side_hits
                if p.parent.name.lower() == side_folder
            }
            if len(exact) == 1:
                return next(iter(exact))
            raise FileNotFoundError(
                f"Multiple {side} keypoint matches for {rel}: "
                f"{[str(p) for p in list(side_hits)[:5]]}"
            )

    if len(hits) == 1:
        return next(iter(hits))
    if len(hits) > 1:
        raise FileNotFoundError(
            f"Asset basename is ambiguous: {rel}\n"
            f"Matches: {[str(p) for p in list(hits)[:8]]}"
        )

    raise FileNotFoundError(
        f"Could not resolve manifest asset: {rel}\n"
        f"Capture root searched: {CAPTURE_DIR}"
    )

with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_data = json.load(f)

if "400485" not in camera_data:
    raise KeyError(f"Camera 400485 not found in {CAMERA_PATH}")

camera = camera_data["400485"]
CAM_R = np.asarray(camera["R"], np.float32)
CAM_T = np.asarray(camera["t"], np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], np.float32).reshape(2)

# -------------------------- Shared model/data setup ---------------------------

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
    12, 13, 14, 15, 16, 17, 19, 20, 21, 22,
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]

def project(points):
    points = np.asarray(points, np.float32)
    z = points[..., 2]
    if np.any(z <= 1.0):
        raise ValueError("A projected hand joint has invalid camera depth.")
    return np.stack(
        (
            CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0],
            CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1],
        ),
        axis=-1,
    ).astype(np.float32)

def crop_affine(uv, margin=1.4, out_w=IMAGE_W, out_h=IMAGE_H):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        np.float32,
    )

def make_channels(gray):
    gray_f = gray.astype(np.float32) / 255.0
    gray_255 = gray.astype(np.float32)
    gx = cv2.Sobel(gray_255, cv2.CV_32F, 1, 0, ksize=3, scale=1 / 1020)
    gy = cv2.Sobel(gray_255, cv2.CV_32F, 0, 1, ksize=3, scale=1 / 1020)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return np.stack((gray_f, mag)).astype(np.float32)

def normalize_uv(uv):
    out = np.empty_like(uv, dtype=np.float32)
    out[..., 0] = 2.0 * uv[..., 0] / IMAGE_W - 1.0
    out[..., 1] = 2.0 * uv[..., 1] / IMAGE_H - 1.0
    return out

def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[..., 0].copy(), points[..., 1].copy()
    out[..., 0] = c * x + s * y
    out[..., 1] = -s * x + c * y
    return out

def augment_one(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)

    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    return make_channels(warped), y3d_aug, normalize_uv(uv_aug)

class WebcamAlignedLCFTracker(nn.Module):
    def __init__(self, canonical_lens):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", canonical_lens.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3), dim=-1, eps=1e-7
        )

        d6 = self.latent_frame_head(feat)
        a1, a2 = d6[:, :3], d6[:, 3:]
        b1 = F.normalize(a1, dim=-1, eps=1e-7)
        b2 = F.normalize(
            a2 - (b1 * a2).sum(-1, keepdim=True) * b1,
            dim=-1,
            eps=1e-7,
        )
        frame = torch.stack(
            (b1, b2, torch.cross(b1, b2, dim=-1)), dim=-1
        )
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(frame, phalanx.transpose(1, 2)).transpose(1, 2)

        dirs = torch.zeros((batch, 20, 3), dtype=x.dtype, device=x.device)
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger, offset in enumerate((4, 8, 12, 16)):
            p_offset = 3 + finger * 3
            dirs[:, offset] = meta[:, finger + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[:, p_offset:p_offset + 3]

        joints = torch.zeros((batch, 21, 3), dtype=x.dtype, device=x.device)
        for k, (parent, child) in enumerate(BONE_PAIRS):
            joints[:, child] = (
                joints[:, parent] + dirs[:, k] * self.bone_lens[k]
            )
        uv = torch.tanh(self.proj2d_head(feat).reshape(batch, 21, 2))
        return joints, dirs, uv

def direct_loss(pred3, target3, pred2, target2):
    parents = [p for p, _ in BONE_PAIRS]
    children = [c for _, c in BONE_PAIRS]
    pred_bones = pred3[:, children] - pred3[:, parents]
    target_bones = target3[:, children] - target3[:, parents]
    pred_u = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_u = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction_loss = (1.0 - (pred_u * target_u).sum(-1)).mean()
    return (
        1.5 * F.l1_loss(pred3, target3)
        + 20.0 * direction_loss
        + 15.0 * F.smooth_l1_loss(pred2, target2)
    )

# -------------------------- Read and cache the data --------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Loading {len(rows):,} camera-400485 rows...")
records, roots, gray_buffers, buffer_affines = [], [], [], []
image_width = image_height = None
missing = []

for i, row in enumerate(rows):
    try:
        kp_path = resolve_asset(row["orig_kp_right"])
        image_path = resolve_asset(row["image_path"])
    except FileNotFoundError as exc:
        missing.append((i, str(exc)))
        if len(missing) >= 5:
            break
        continue

    with kp_path.open(encoding="utf-8") as f:
        world = np.asarray(json.load(f), dtype=np.float32)

    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"OpenCV could not read image: {image_path}")

    h, w = gray.shape
    if image_width is None:
        image_width, image_height = w, h
    elif (w, h) != (image_width, image_height):
        raise ValueError("Dataset contains inconsistent image dimensions.")

    camera_pts = world @ CAM_R.T + CAM_T
    selected = camera_pts[KEEP_21_INDICES]
    uv_image = project(selected)
    y3d = selected - selected[0:1]

    affine = crop_affine(uv_image, margin=1.4)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = uv_image @ affine[:, :2].T + affine[:, 2]

    buffer_affine = crop_affine(
        uv_image, margin=3.5, out_w=384, out_h=288
    )
    gray_buffer = cv2.warpAffine(
        gray, buffer_affine, (384, 288),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    records.append({
        "frame_id": row.get("frame_id", str(i)),
        "segment": row["segment"],
        "uv_image": uv_image,
        "uv_crop": uv_crop.astype(np.float32),
        "crop_gray": crop_gray,
        "y3d": y3d.astype(np.float32),
    })
    roots.append(selected[0].copy())
    gray_buffers.append(gray_buffer)
    buffer_affines.append(buffer_affine)

if missing:
    details = "\n".join(f"row {i}: {msg}" for i, msg in missing)
    raise FileNotFoundError(
        "Some manifest assets could not be resolved. First failures:\n" + details
    )
if not records:
    raise RuntimeError("No usable records were loaded.")

roots = np.asarray(roots, np.float32)
gray_buffers = np.stack(gray_buffers)
buffer_affines = np.stack(buffer_affines)
y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
x_all = torch.from_numpy(
    np.stack([make_channels(r["crop_gray"]) for r in records])
)

print(f"Loaded {len(records):,} frames at {image_width}x{image_height}.")

# ----------------------- Deterministic segment split -------------------------

segments = np.asarray(sorted({r["segment"] for r in records}), dtype=object)
if len(segments) != TRAIN_SEGMENTS + VAL_SEGMENTS:
    raise ValueError(
        f"Expected {TRAIN_SEGMENTS + VAL_SEGMENTS} segments; found {len(segments)}."
    )

split_rng = np.random.default_rng(SEED)
split_rng.shuffle(segments)
train_segments = set(segments[:TRAIN_SEGMENTS].tolist())
val_segments = set(segments[TRAIN_SEGMENTS:].tolist())

train_idx = np.asarray(
    [i for i, r in enumerate(records) if r["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, r in enumerate(records) if r["segment"] in val_segments],
    dtype=np.int64,
)
assert not train_segments.intersection(val_segments)

print(
    f"Split: {len(train_segments)} train / {len(val_segments)} val segments; "
    f"{len(train_idx):,} train / {len(val_idx):,} val frames."
)

def score_pose(pred, target):
    distances = np.linalg.norm(pred - target, axis=-1)
    return float(distances.mean()), [
        float(distances[:, j].mean()) for j in FINGERTIP_JOINTS
    ]

def format_tips(tips):
    return " ".join(
        f"{name}={value:.2f}"
        for name, value in zip(FINGER_NAMES, tips)
    )

def canonical_bone_lengths(ids):
    return torch.from_numpy(np.stack([
        np.linalg.norm(
            y_all[ids, child] - y_all[ids, parent], axis=1
        ).mean()
        for parent, child in BONE_PAIRS
    ]).astype(np.float32))

def predict_direct(model, ids):
    model.eval()
    predictions = []
    with torch.inference_mode():
        for start in range(0, len(ids), 128):
            chunk = ids[start:start + 128]
            pred, _, _ = model(x_all[chunk].to(DEVICE))
            predictions.append(pred.cpu().numpy())
    return np.concatenate(predictions, axis=0)

# ---------------------- Old checkpoint discovery/load ------------------------

def find_old_checkpoint():
    found = []
    for root in (Path("/content"), Path.cwd(), DRIVE_ROOT):
        if not root.exists():
            continue
        for name in CHECKPOINT_NAMES:
            found.extend(root.rglob(name))

    if not found:
        return None

    for name in CHECKPOINT_NAMES:
        match = next((p for p in found if p.name == name), None)
        if match is not None:
            return match
    return found[0]

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def strip_module_prefix(state):
    if not isinstance(state, dict):
        return state
    return {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in state.items()
    }

class OldPoseRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.delta_head = nn.Linear(512, 63)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        return self.delta_head(feat).reshape(-1, 21, 3)

old_checkpoint_path = find_old_checkpoint()
old_direct, old_refiner = None, None
old_val_pred = None

if old_checkpoint_path is None:
    print(
        "\nNo old checkpoint found under /content, current folder, or Drive. "
        "The new OOF experiment will train from scratch."
    )
else:
    print(f"\nFound old checkpoint: {old_checkpoint_path}")
    checkpoint = load_checkpoint(old_checkpoint_path)

    if isinstance(checkpoint, dict):
        direct_state = checkpoint.get("direct_state_dict")
        if direct_state is None and "state_dict" in checkpoint:
            direct_state = checkpoint["state_dict"]

        bone_lens = checkpoint.get("bone_lens")
        if bone_lens is None:
            bone_lens = canonical_bone_lengths(train_idx)
        else:
            bone_lens = torch.as_tensor(bone_lens, dtype=torch.float32).cpu()

        if direct_state is not None:
            old_direct = WebcamAlignedLCFTracker(bone_lens).to(DEVICE)
            try:
                old_direct.load_state_dict(
                    strip_module_prefix(direct_state), strict=True
                )
                old_direct.eval()
                print(f"Loaded old direct model onto {DEVICE}.")
            except Exception as exc:
                old_direct = None
                print(f"Old direct weights did not match this architecture: {exc}")

        refiner_state = checkpoint.get("refiner_state_dict")
        if refiner_state is not None:
            old_refiner = OldPoseRefiner().to(DEVICE)
            try:
                old_refiner.load_state_dict(
                    strip_module_prefix(refiner_state), strict=True
                )
                old_refiner.eval()
                print(f"Loaded old Phase 1 refiner onto {DEVICE}.")
            except Exception as exc:
                old_refiner = None
                print(f"Old refiner weights did not match: {exc}")

if old_direct is not None:
    old_val_pred = predict_direct(old_direct, val_idx)
    old_mpjpe, old_tips = score_pose(old_val_pred, y_all[val_idx])
    print(
        f"Old checkpoint direct | val {old_mpjpe:.2f} mm | "
        f"{format_tips(old_tips)}"
    )

# --------------------- Prior crop/render helper functions --------------------

def prior_input(index, prior):
    uv_img = project(roots[index:index + 1] + prior)
    affine = crop_affine(uv_img, margin=1.4)

    src = np.eye(3, dtype=np.float32)
    src[:2] = buffer_affines[index]
    dst = np.eye(3, dtype=np.float32)
    dst[:2] = affine
    mapping = dst @ np.linalg.inv(src)

    gray = cv2.warpAffine(
        gray_buffers[index],
        mapping[:2],
        (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    xy = uv_img @ affine[:, :2].T + affine[:, 2]
    lines = np.zeros((IMAGE_H, IMAGE_W), np.uint8)
    dots = np.zeros_like(lines)
    points = np.rint(xy).astype(int)

    for parent, child in BONE_PAIRS:
        cv2.line(
            lines, tuple(points[parent]), tuple(points[child]),
            255, 2, cv2.LINE_AA,
        )
    for point in points:
        cv2.circle(dots, tuple(point), 3, 255, -1, cv2.LINE_AA)

    inp = np.concatenate((
        make_channels(gray),
        lines[None].astype(np.float32) / 255.0,
        dots[None].astype(np.float32) / 255.0,
    ))
    return inp, affine

# Evaluate old Phase 1 weights, if present, on the old direct model's predictions.
if old_direct is not None and old_refiner is not None:
    print("\nOld checkpoint Phase 1 iterative validation:")
    current = old_val_pred.copy()
    for iteration in range(4):
        mpjpe, tips = score_pose(current, y_all[val_idx])
        print(
            f"  Iteration {iteration}: {mpjpe:.2f} mm | "
            f"{format_tips(tips)}"
        )
        if iteration < 3:
            updated = []
            with torch.inference_mode():
                for idx, prior in zip(val_idx, current):
                    inp, _ = prior_input(int(idx), prior)
                    delta = old_refiner(
                        torch.from_numpy(inp[None]).to(DEVICE)
                    )[0].cpu().numpy()
                    updated.append(prior + delta)
            current = np.asarray(updated, np.float32)

# ----------------------- Train direct models for OOF -------------------------

def train_direct(model, train_ids, seed_offset=0):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
    )
    rng = np.random.default_rng(SEED + 1 + seed_offset)

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        order = np.random.permutation(train_ids)

        for start in range(0, len(order), BATCH_SIZE):
            ids = order[start:start + BATCH_SIZE]
            samples = [
                augment_one(
                    records[int(i)]["crop_gray"],
                    records[int(i)]["uv_crop"],
                    y_all[int(i)],
                    rng,
                )
                for i in ids
            ]
            bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
            by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
            by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred3, _, pred2 = model(bx)
            loss = direct_loss(pred3, by3, pred2, by2)
            loss.backward()
            optimizer.step()

        scheduler.step()

    return model

fold_rng = np.random.default_rng(SEED + 300)
shuffled_train_segments = np.asarray(sorted(train_segments), dtype=object)
fold_rng.shuffle(shuffled_train_segments)
fold_groups = np.array_split(shuffled_train_segments, N_FOLDS)

oof_train = np.full((len(records), 21, 3), np.nan, np.float32)
val_fold_preds = []
fold_states = []
fold_models = []

for fold_id, group in enumerate(fold_groups):
    held_segments = set(group.tolist())
    fold_val = np.asarray(
        [i for i in train_idx if records[int(i)]["segment"] in held_segments],
        dtype=np.int64,
    )
    fold_train = np.asarray(
        [i for i in train_idx if records[int(i)]["segment"] not in held_segments],
        dtype=np.int64,
    )
    assert not held_segments.intersection(
        records[int(i)]["segment"] for i in fold_train
    )

    lens = canonical_bone_lengths(fold_train)
    torch.manual_seed(SEED + 1000 + fold_id)
    model = WebcamAlignedLCFTracker(lens).to(DEVICE)
    train_direct(model, fold_train, seed_offset=fold_id * 100)

    oof_train[fold_val] = predict_direct(model, fold_val)
    val_fold_preds.append(predict_direct(model, val_idx))
    fold_states.append({
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    })
    fold_models.append(model)
    print(
        f"Finished OOF fold {fold_id + 1}/{N_FOLDS}: "
        f"{len(fold_train):,} train, {len(fold_val):,} held-out frames."
    )

assert np.isfinite(oof_train[train_idx]).all()
val_prior = np.mean(np.stack(val_fold_preds), axis=0).astype(np.float32)
oof_mpjpe, oof_tips = score_pose(val_prior, y_all[val_idx])
print(
    f"OOF direct ensemble | val {oof_mpjpe:.2f} mm | "
    f"{format_tips(oof_tips)}"
)

# ---------------------- Augmented prior training samples ---------------------

def make_augmented_prior_sample(index, prior, rng):
    inp, affine = prior_input(index, prior)
    gt = records[index]["y3d"]
    gt_uv_crop = records[index]["uv_image"] @ affine[:, :2].T + affine[:, 2]

    gray_crop = np.clip(inp[0] * 255.0, 0, 255).astype(np.uint8)
    base, gt_aug, uv_target = augment_one(
        gray_crop, gt_uv_crop, gt, rng
    )

    uv_pixels = uv_target.copy()
    uv_pixels[:, 0] = (uv_pixels[:, 0] + 1.0) * IMAGE_W * 0.5
    uv_pixels[:, 1] = (uv_pixels[:, 1] + 1.0) * IMAGE_H * 0.5

    design = np.column_stack((gt_uv_crop, np.ones(len(gt_uv_crop))))
    coeff, *_ = np.linalg.lstsq(design, uv_pixels, rcond=None)
    aug_affine = coeff.T.astype(np.float32)
    angle = float(np.arctan2(aug_affine[0, 1], aug_affine[0, 0]))
    prior_aug = rotate_camera_xy(prior, angle).astype(np.float32)

    lines = cv2.warpAffine(
        inp[2], aug_affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )
    dots = cv2.warpAffine(
        inp[3], aug_affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )
    x4 = np.concatenate((base, lines[None], dots[None]), axis=0)

    return (
        x4, prior_aug, gt_aug, uv_target,
        roots[index], affine, aug_affine,
    )

# ------------------------- Train gated refiner -------------------------------

class GatedRefiner(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.delta_head = nn.Linear(512, 63)
        self.uncertainty_head = nn.Linear(512, 21)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        delta = 45.0 * torch.tanh(
            self.delta_head(feat).reshape(-1, 21, 3) / 45.0
        )
        sigma = (
            F.softplus(self.uncertainty_head(feat).reshape(-1, 21)) + 1.0
        ).clamp(1.0, 60.0)
        authority = 20.0 / (20.0 + sigma)
        return delta, sigma, authority

gated = GatedRefiner().to(DEVICE)
optimizer = torch.optim.AdamW(
    gated.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=REFINER_EPOCHS, eta_min=1e-4
)
rng = np.random.default_rng(SEED + 1300)
focal_t = torch.as_tensor(CAM_FOCAL, dtype=torch.float32, device=DEVICE)
principal_t = torch.as_tensor(CAM_PRINCPT, dtype=torch.float32, device=DEVICE)

for epoch in range(1, REFINER_EPOCHS + 1):
    gated.train()
    order = np.random.permutation(train_idx)

    for start in range(0, len(order), BATCH_SIZE):
        ids = order[start:start + BATCH_SIZE]
        samples = [
            make_augmented_prior_sample(
                int(i), oof_train[int(i)], rng
            )
            for i in ids
        ]

        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        prior_b = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        target_b = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)
        uv_target = torch.from_numpy(np.stack([s[3] for s in samples])).to(DEVICE)
        roots_b = torch.from_numpy(np.stack([s[4] for s in samples])).to(DEVICE)
        crop_aff = torch.from_numpy(np.stack([s[5] for s in samples])).to(DEVICE)
        aug_aff = torch.from_numpy(np.stack([s[6] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        delta, sigma, authority = gated(bx)
        corrected = prior_b + authority[..., None] * delta

        pose_loss = F.l1_loss(corrected, target_b)
        joint_error = torch.linalg.vector_norm(
            corrected - target_b, dim=-1
        )
        uncertainty_loss = (
            joint_error / sigma + torch.log(sigma)
        ).mean()

        angle = torch.atan2(aug_aff[:, 0, 1], aug_aff[:, 0, 0])
        c, s = torch.cos(angle)[:, None], torch.sin(angle)[:, None]
        unrot = corrected.clone()
        x, y = corrected[..., 0], corrected[..., 1]
        unrot[..., 0] = c * x - s * y
        unrot[..., 1] = s * x + c * y

        camera_pts = unrot + roots_b[:, None, :]
        z = camera_pts[..., 2].clamp_min(1.0)
        uv = torch.stack((
            focal_t[0] * camera_pts[..., 0] / z + principal_t[0],
            focal_t[1] * camera_pts[..., 1] / z + principal_t[1],
        ), dim=-1)

        uv_h = torch.cat((uv, torch.ones_like(uv[..., :1])), dim=-1)
        crop_xy = torch.bmm(uv_h, crop_aff.transpose(1, 2))
        crop_h = torch.cat((crop_xy, torch.ones_like(crop_xy[..., :1])), dim=-1)
        aug_xy = torch.bmm(crop_h, aug_aff.transpose(1, 2))

        uv_pred = aug_xy.clone()
        uv_pred[..., 0] = 2.0 * aug_xy[..., 0] / IMAGE_W - 1.0
        uv_pred[..., 1] = 2.0 * aug_xy[..., 1] / IMAGE_H - 1.0
        reprojection_loss = F.smooth_l1_loss(uv_pred, uv_target)

        loss = pose_loss + 0.25 * uncertainty_loss + 15.0 * reprojection_loss
        loss.backward()
        optimizer.step()

    scheduler.step()
    print(f"Refiner epoch {epoch}/{REFINER_EPOCHS} complete")

# --------------------------- Validation and timing ---------------------------

gated.eval()

def apply_refiner(index, prior):
    inp, _ = prior_input(int(index), prior)
    with torch.inference_mode():
        delta, _, authority = gated(
            torch.from_numpy(inp[None]).to(DEVICE)
        )
    return (
        prior + (authority[0, :, None] * delta[0]).cpu().numpy()
    ).astype(np.float32)

eval_rows = []
current = val_prior.copy()
for iteration in range(4):
    mpjpe, tips = score_pose(current, y_all[val_idx])
    eval_rows.append((iteration, mpjpe, tips))
    if iteration < 3:
        current = np.stack([
            apply_refiner(int(i), pose)
            for i, pose in zip(val_idx, current)
        ])

latency_idx = int(val_idx[0])
latency_prior = val_prior[0].copy()

def one_refiner_step():
    return apply_refiner(latency_idx, latency_prior)

def direct_ensemble_plus_two():
    image = x_all[latency_idx:latency_idx + 1].to(DEVICE)
    with torch.inference_mode():
        poses = [model(image)[0][0] for model in fold_models]
        prior = torch.stack(poses).mean(dim=0).cpu().numpy()
    for _ in range(2):
        prior = apply_refiner(latency_idx, prior)
    return prior

for _ in range(20):
    one_refiner_step()
    direct_ensemble_plus_two()

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

one_ms, combo_ms = [], []
for _ in range(200):
    start = time.perf_counter()
    one_refiner_step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    one_ms.append((time.perf_counter() - start) * 1000.0)

    start = time.perf_counter()
    direct_ensemble_plus_two()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    combo_ms.append((time.perf_counter() - start) * 1000.0)

# --------------------------------- Checkpoint --------------------------------

checkpoint_out = Path("/content/phase2_oof_gated_refiner.pt")
torch.save(
    {
        "fold_direct_state_dicts": fold_states,
        "gated_refiner_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in gated.state_dict().items()
        },
        "fold_segments": [list(group) for group in fold_groups],
        "train_segments": sorted(train_segments),
        "val_segments": sorted(val_segments),
        "seed": SEED,
        "camera_id": "400485",
        "manifest_path": str(MANIFEST_PATH),
    },
    checkpoint_out,
)

print("\nOOF-TRAINED GATED REFINEMENT — HELD-OUT VALIDATION")
print(f"{'Iteration':<12} {'MPJPE mm':>9}  Thumb   Index  Middle    Ring   Pinky")
print("-" * 75)
for iteration, mpjpe, tips in eval_rows:
    print(
        f"{iteration:<12} {mpjpe:9.2f}  "
        + " ".join(f"{value:7.2f}" for value in tips)
    )

print(
    f"\nMedian batch-1 latency on {DEVICE}: "
    f"refiner={statistics.median(one_ms):.3f} ms; "
    f"direct ensemble + 2 refinements="
    f"{statistics.median(combo_ms):.3f} ms"
)
print(f"Checkpoint saved: {checkpoint_out}")

In [ ]:
# @title
# Damped-refiner diagnostic. Uses variables from the previous runner.

expected_shape = (len(val_idx), 21, 3)
base_prior = np.asarray(val_prior, dtype=np.float32)

# Recover the validation prior if the notebook variable was overwritten.
if base_prior.shape != expected_shape:
    if "val_fold_preds" in globals() and len(val_fold_preds):
        base_prior = np.mean(
            np.stack(val_fold_preds), axis=0
        ).astype(np.float32)
    elif "fold_models" in globals() and len(fold_models):
        base_prior = np.mean(
            np.stack([predict_direct(model, val_idx) for model in fold_models]),
            axis=0,
        ).astype(np.float32)
    else:
        raise ValueError(
            f"Expected val_prior shape {expected_shape}, got "
            f"{base_prior.shape}; cannot rebuild it from available variables."
        )

if base_prior.shape != expected_shape:
    raise ValueError(
        f"Validation prior has shape {base_prior.shape}; expected {expected_shape}."
    )

alphas = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 1.0]
diagnostic_rows = []

for alpha in alphas:
    current = base_prior.copy()

    for step in range(4):
        mpjpe, tips = score_pose(current, y_all[val_idx])
        diagnostic_rows.append((alpha, step, mpjpe, tips))

        if step < 3 and alpha > 0:
            # apply_refiner returns a complete (21, 3) pose; do not index [0].
            proposed = np.stack([
                apply_refiner(int(i), pose)
                for i, pose in zip(val_idx, current)
            ]).astype(np.float32)

            if proposed.shape != expected_shape:
                raise ValueError(
                    f"Refiner returned shape {proposed.shape}; "
                    f"expected {expected_shape}."
                )

            current = current + alpha * (proposed - current)

print("DAMPED REFINER DIAGNOSTIC — VALIDATION SEGMENTS")
print(f"{'Alpha':>6} {'Step':>6} {'MPJPE mm':>10}  Thumb   Index  Middle    Ring   Pinky")
print("-" * 78)
for alpha, step, mpjpe, tips in diagnostic_rows:
    print(
        f"{alpha:6.2f} {step:6d} {mpjpe:10.2f}  "
        + " ".join(f"{value:7.2f}" for value in tips)
    )